# Simple Random Forest + Optuna search

Same structure as the shared Arsenal + Optuna notebook: constants, loading, an Optuna objective, and SQLite storage so the study can be resumed.

Our version uses the final model's features (from notebook 10) and the road split: each road is predicted by a model trained on the other two. The score is false alarm spots per km while finding at least 90% of the bumps (lower = better).

**Important:** Optuna chooses the settings using these three roads, so the best score is optimistic. The winner is checked afterwards with 3 seeds and the strict test.

In [4]:
import sys
!"{sys.executable}" -m pip install optuna


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import joblib
import numpy as np
import pandas as pd
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, LeaveOneGroupOut
from data_utils import add_model_features, spots_at_goal

# We're going to set the logging of OPTUNA to on.
# That way we can easily monitor progress as it proceeds.
optuna.logging.set_verbosity(optuna.logging.INFO)

c:\D_DRIVE\Roadscanners\SemesterProjectFall2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### CONSTANTS

In [2]:
# Here is our data and our final model
WINDOWS_FILE = "data_features/windows_5m.csv"
MODEL_FILE = "models/rf_final.joblib"

# The same features as our final model (relative shaking, tilt, neighbours ±2)
FEATURES = joblib.load(MODEL_FILE)["features"]
TARGET = "is_bump"

# Optuna search ranges
# We can ADJUST THESE!
N_ESTIMATORS_RANGE = (100, 500)
N_ESTIMATORS_STEP = 100

MIN_SAMPLES_LEAF_RANGE = (1, 20)

# None = no limit
MAX_DEPTH_CHOICES = [None, 5, 10, 20] 
# features tried per split          
MAX_FEATURES_CHOICES = ["sqrt", "log2", None]  
CLASS_WEIGHTS = ["balanced", "balanced_subsample"]

# CPU usage (for when we still need to use our PC while trials are happening)
CPU_LIMIT = 2

# Increase the number of experiments here to set how long the session will run.
N_TRIALS = 50
SEED = 101

# SQLite
STUDY_NAME = "frostbumps_rf_simple"
STORAGE = "sqlite:///frostbumps_rf_simple.db"

### LOAD DATA

1. Loads the 5 m windows.
2. Adds the final model's features, one drive at a time (same function as the dashboard).
3. Removes the unclear "left out" windows and makes the target.
4. X, y and the road of each window (used for the road split).

In [3]:
all_windows = pd.read_csv(WINDOWS_FILE)

# Add the features one drive at a time
parts = []
for drive, group in all_windows.groupby("drive"):
    parts.append(add_model_features(group))
all_windows = pd.concat(parts)

# Keep only clear windows and make the target
windows = all_windows[all_windows["label"] != "left out"].copy()
windows[TARGET] = (windows["label"] == "bump").astype(int)

# X, y split + the road of each window
X = windows[FEATURES]
y = windows[TARGET]
roads = windows["road"]

print(X.shape, "| roads:", list(roads.unique()))

(2728, 40) | roads: ['Iisingintie', 'Kaittiaisentie', 'sinetänsalmentie']


### OPTUNA objective

Trains the random forest with the suggested settings. `LeaveOneGroupOut` is the road split: each road is predicted by a model trained on the other two.

In [4]:
def objective(trial):

    # This runs your suggestion parameters (which were constants in the cells above)
    n_estimators = trial.suggest_int("n_estimators", *N_ESTIMATORS_RANGE, step=N_ESTIMATORS_STEP)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", *MIN_SAMPLES_LEAF_RANGE)
    max_depth = trial.suggest_categorical("max_depth", MAX_DEPTH_CHOICES)
    max_features = trial.suggest_categorical("max_features", MAX_FEATURES_CHOICES)
    class_weight = trial.suggest_categorical("class_weight", CLASS_WEIGHTS)

    # The random forest
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        max_depth=max_depth,
        max_features=max_features,
        class_weight=class_weight,
        random_state=SEED,
        n_jobs=CPU_LIMIT,
    )

    # Road split: every road is predicted by a model trained on the other two
    prob = cross_val_predict(model, X, y, groups=roads, cv=LeaveOneGroupOut(),
                             method="predict_proba")[:, 1]

    # Score the trial: false alarm spots per km while finding at least 90% of the bumps
    score = spots_at_goal(windows, prob)

    return score

### RUN / RESUME OPTUNA

In [9]:
# Run / interrupt this cell freely as needed.
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True,
)

# The first trial is our current model from notebook 10 (the number to beat)
if len(study.trials) == 0:
    study.enqueue_trial({"n_estimators": 300, "min_samples_leaf": 5, "max_depth": None,
                         "max_features": "sqrt", "class_weight": "balanced"})

study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

print("\nBest false alarm spots per km:", round(study.best_value, 2))
print("Best parameters:")
for parameter, value in study.best_params.items():
    print(f"{parameter}: {value}")

[I 2026-11-24 17:25:15,935] A new study created in RDB with name: frostbumps_rf_simple
[I 2026-11-24 17:25:18,050] Trial 0 finished with value: 3.2991202346041053 and parameters: {'n_estimators': 300, 'min_samples_leaf': 5, 'max_depth': None, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 0 with value: 3.2991202346041053.
[I 2026-11-24 17:25:20,283] Trial 1 finished with value: 2.6392961876832843 and parameters: {'n_estimators': 300, 'min_samples_leaf': 12, 'max_depth': 20, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 1 with value: 2.6392961876832843.
[I 2026-11-24 17:25:21,584] Trial 2 finished with value: 2.7126099706744866 and parameters: {'n_estimators': 200, 'min_samples_leaf': 4, 'max_depth': 5, 'max_features': 'log2', 'class_weight': 'balanced'}. Best is trial 1 with value: 2.6392961876832843.
[I 2026-11-24 17:25:23,633] Trial 3 finished with value: 3.885630498533724 and parameters: {'n_estimators': 100, 'min_samples_leaf': 3, 


Best false alarm spots per km: 1.76
Best parameters:
n_estimators: 500
min_samples_leaf: 6
max_depth: 5
max_features: log2
class_weight: balanced


## Viewing previous trials

In [5]:
study = optuna.load_study(study_name=STUDY_NAME, storage=STORAGE)

In [6]:
results = study.trials_dataframe(attrs=("number", "value", "params", "state"))
results = results.sort_values("value", ascending=True)
display(results.head(10))

,number,value,params_class_weight,params_max_depth,params_max_features,params_min_samples_leaf,params_n_estimators,state
47,47,1.759531,balanced,5.0,log2,6,500,COMPLETE
43,43,1.832845,balanced,5.0,log2,6,300,COMPLETE
20,20,1.906158,balanced,5.0,log2,9,300,COMPLETE
30,30,1.979472,balanced,5.0,log2,7,500,COMPLETE
48,48,1.979472,balanced,5.0,log2,7,500,COMPLETE
37,37,1.979472,balanced_subsample,5.0,log2,6,300,COMPLETE
22,22,2.052786,balanced,5.0,sqrt,8,300,COMPLETE
25,25,2.052786,balanced,5.0,log2,7,300,COMPLETE
32,32,2.126100,balanced,5.0,log2,8,100,COMPLETE
14,14,2.199413,balanced,5.0,log2,14,400,COMPLETE


In [9]:
# Let's see how our current model scores is
print("current model (trial 0):", round(study.trials[0].value, 2))

current model (trial 0): 3.3


In [10]:
# Let's check with different seeds to see how stable the results are.
def check_params(params, seeds=[1, 2, 3]):
    scores = []
    for seed in seeds:
        model = RandomForestClassifier(**params, random_state=seed, n_jobs=CPU_LIMIT)
        prob = cross_val_predict(model, X, y, groups=roads, cv=LeaveOneGroupOut(),
                                 method="predict_proba")[:, 1]
        scores.append(spots_at_goal(windows, prob))
    return scores


current = {"n_estimators": 300, "min_samples_leaf": 5, "max_depth": None,
           "max_features": "sqrt", "class_weight": "balanced"}
best = study.best_params

current_scores = check_params(current)
best_scores = check_params(best)
print("current model:", [round(s, 2) for s in current_scores], "| average", round(np.mean(current_scores), 2))
print("Optuna best:  ", [round(s, 2) for s in best_scores], "| average", round(np.mean(best_scores), 2))

current model: [2.71, 3.3, 3.15] | average 3.05
Optuna best:   [2.71, 2.05, 2.64] | average 2.47


* Never worse, better for 2 of 3 seeds, and also more stable. So it's a real, modest improvement, not luck.

### Finally checking with the strict test

In [11]:
from data_utils import bump_level, threshold_at_goal

def strict_test(params, seed):
    parts = []
    for test_road in roads.unique():
        train = windows[roads != test_road]
        test = windows[roads == test_road]

        # 1. Practice predictions: the two training roads take turns
        model = RandomForestClassifier(**params, random_state=seed, n_jobs=CPU_LIMIT)
        practice_prob = cross_val_predict(model, train[FEATURES], train[TARGET], groups=train["road"],
                                          cv=LeaveOneGroupOut(), method="predict_proba")[:, 1]

        # 2. The threshold is chosen on the practice predictions only
        t = threshold_at_goal(train, practice_prob)

        # 3. Train on both training roads, apply that threshold to the test road
        model.fit(train[FEATURES], train[TARGET])
        p = test[["drive", "window", TARGET]].copy()
        p["pred"] = (model.predict_proba(test[FEATURES])[:, 1] >= t).astype(int)
        parts.append(p)

    return bump_level(pd.concat(parts))


for name, params in [("current model", current), ("Optuna best", best)]:
    found = []
    spots = []
    for seed in [1, 2, 3]:
        r = strict_test(params, seed)
        found.append(r["bumps_found"])
        spots.append(r["spots_per_km"])
    print(name, "| found", round(np.mean(found), 1), "of 48 | spots/km", round(np.mean(spots), 2))

current model | found 47.7 of 48 | spots/km 8.53
Optuna best | found 48.0 of 48 | spots/km 4.91


* Under the strict test, the Optuna settings found all bumps with almost half the false alarms.



### Will use Optuna settings for the final model

In [12]:
import shutil

# Keep the old model and predictions as a backup
shutil.copy("models/rf_final.joblib", "models/rf_final_v1_notebook10.joblib")
shutil.copy("predictions/predictions_rf_final.csv", "predictions/predictions_rf_final_v1_notebook10.csv")

FINAL_SEED = 101
params = best

# 1. Cross-validated predictions: every road predicted by a model that never saw it
model = RandomForestClassifier(**params, random_state=FINAL_SEED, n_jobs=CPU_LIMIT)
prob = cross_val_predict(model, X, y, groups=roads, cv=LeaveOneGroupOut(), method="predict_proba")[:, 1]

final_threshold = threshold_at_goal(windows, prob)
cv_pred = windows[["drive", "road", "window", TARGET]].copy()
cv_pred["prob"] = prob
cv_pred["pred"] = (prob >= final_threshold).astype(int)
cv_pred.to_csv("predictions/predictions_rf_final.csv", index=False)

print("default threshold:", round(final_threshold, 2))
print(bump_level(cv_pred))

# 2. The final model on all roads, plus one model per road (each trained without that road)
final_model = RandomForestClassifier(**params, random_state=FINAL_SEED, n_jobs=CPU_LIMIT)
final_model.fit(X, y)

road_models = {}
for road in roads.unique():
    m = RandomForestClassifier(**params, random_state=FINAL_SEED, n_jobs=CPU_LIMIT)
    m.fit(X[roads != road], y[roads != road])
    road_models[road] = m

joblib.dump({"model": final_model, "features": FEATURES, "threshold": final_threshold,
             "road_models": road_models, "params": params},
            "models/rf_final.joblib")
print("saved: models/rf_final.joblib")

default threshold: 0.65
{'bumps_total': 48, 'bumps_found': 44, 'detection_rate': 0.9166666666666666, 'real_false_alarms': 77, 'false_alarms_per_km': 5.64516129032258, 'spots_per_km': 1.7595307917888563}
saved: models/rf_final.joblib
